In [ ]:
import logging
import re
import sys

import boto3
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import functions as F

In [ ]:
# ##################################################################################################
# Parâmetros do Job — usados em produção pelo Glue (comentado no ambiente local)
#
# args = getResolvedOptions(
#     sys.argv, ["JOB_NAME", "PIPELINE_NAME", "LANDING_ZONE_BUCKET", "BRONZE_DATABASE"]
# )
#
# PIPELINE_NAME = args["PIPELINE_NAME"]
# LANDING_ZONE_BUCKET = args["LANDING_ZONE_BUCKET"]
# BRONZE_DATABASE = args["BRONZE_DATABASE"]

In [ ]:
# ##################################################################################################
# Variáveis locais — substitui os parâmetros do job para execução no notebook

PIPELINE_NAME       = "health_insurance"
LANDING_ZONE_BUCKET = "637423524537-eedb015-landing"
BRONZE_DATABASE     = "eedb015_bronze"

# Limite de tabelas a processar — útil para não sobrecarregar o ambiente local.
# Altere para None para processar todas.
LOCAL_TABLE_LIMIT = 3

In [ ]:
# ##################################################################################################
# Inicialização Spark / Glue com extensões Iceberg

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s — %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
)
logger = logging.getLogger("landing_to_bronze")

scf = SparkConf()
scf.setAll([
    ("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"),
    ("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog"),
    # ("spark.sql.catalog.glue_catalog.warehouse", f"s3://<bronze-bucket>/"),
    ("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog"),
    ("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO"),
    ("spark.sql.defaultCatalog", "glue_catalog"),
    # Parser de datas legado (compatibilidade com dados históricos)
    ("spark.sql.legacy.timeParserPolicy", "LEGACY"),
    # Otimização de escrita no S3
    ("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2"),
    ("spark.speculation", "true"),
    # Sobrescrita dinâmica de partições
    ("spark.sql.sources.partitionOverwriteMode", "dynamic"),
])

sc = SparkContext(conf=scf)
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session

# Em produção o job é inicializado via:
# job = Job(glue_ctx)
# job.init(args['JOB_NAME'], args)

In [ ]:
# ##################################################################################################
# Funções auxiliares: mapeamento de caminho S3 → nome de tabela Glue

def camel_to_snake(name: str) -> str:
    """
    Converte CamelCase para snake_case.
    Exemplo: "BenefitsCostSharing" → "benefits_cost_sharing"
    """
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()


def s3_path_to_table_name(s3_key: str) -> str:
    """
    Converte um caminho S3 em nome de tabela snake_case, removendo o prefixo
    do pipeline para manter nomes curtos e legíveis no Glue Catalog.

    Exemplos (com PIPELINE_NAME="health_insurance"):
      raw/health_insurance/Rate.csv              → rate
      raw/health_insurance/PlanAttributes.csv    → plan_attributes
      raw/health_insurance/raw/2014/Rate_PUF.csv → raw_2014_rate_puf
    """
    # Remove extensão .csv
    path_without_ext = re.sub(r"\.csv$", "", s3_key, flags=re.IGNORECASE)

    # Divide o caminho em partes e converte cada uma para snake_case
    parts = path_without_ext.split("/")
    snake_parts = [camel_to_snake(part) for part in parts]

    table_name = "_".join(snake_parts)
    table_name = table_name.replace("-", "_")
    # Colapsa underscores consecutivos (ex: duplo __ vindo de datas)
    table_name = re.sub(r"_+", "_", table_name)

    # Remove o prefixo do pipeline para encurtar o nome da tabela
    pipeline_snake = camel_to_snake(PIPELINE_NAME)
    prefix_to_remove = f"raw_{pipeline_snake}_"
    table_name = table_name.replace(prefix_to_remove, "")

    return table_name


# Teste rápido das funções
_samples = [
    "raw/health_insurance/Rate.csv",
    "raw/health_insurance/PlanAttributes.csv",
    "raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv",
    "raw/health_insurance/raw/2016/ServiceArea_PUF_2015-12-08.csv",
]
for s in _samples:
    print(f"{s:60s} → {s3_path_to_table_name(s)}")

In [ ]:
# ##################################################################################################
# Descoberta dos arquivos CSV na Landing Zone (prefixo: raw/)

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

# file_groups: table_name → lista de s3_paths
file_groups: dict[str, list[str]] = {}

for page in paginator.paginate(Bucket=LANDING_ZONE_BUCKET, Prefix="raw/"):
    for obj in page.get("Contents", []):
        key = obj["Key"]

        # Filtra apenas arquivos CSV
        if not key.lower().endswith(".csv"):
            continue

        table_name = s3_path_to_table_name(key)
        s3_path = f"s3://{LANDING_ZONE_BUCKET}/{key}"
        file_groups.setdefault(table_name, []).append(s3_path)

        print(f"{key:60s} → {table_name}")

logger.info("Total de tabelas descobertas: %d", len(file_groups))

In [ ]:
# Inspeciona os primeiros grupos para validar o mapeamento antes de escrever
list(file_groups.items())[:5]

In [ ]:
# ##################################################################################################
# Processamento: uma tabela Iceberg por tipo de arquivo
#
# LOCAL_TABLE_LIMIT controla quantas tabelas são processadas neste notebook.
# Em produção (landing_to_bronze.py) todas as tabelas são sempre processadas.

tables_to_process = sorted(file_groups.items())
if LOCAL_TABLE_LIMIT:
    logger.info("Ambiente local: limitando a %d tabela(s).", LOCAL_TABLE_LIMIT)
    tables_to_process = tables_to_process[:LOCAL_TABLE_LIMIT]

tables_ok: list[str] = []
tables_failed: list[str] = []

for table_name, s3_paths in tables_to_process:
    bronze_table = f"{BRONZE_DATABASE}.{table_name}"
    logger.info("Processando tabela '%s' (%d arquivo(s))…", bronze_table, len(s3_paths))

    try:
        df = (
            spark.read.csv(
                path=s3_paths,
                header=True,
                inferSchema=False,
                encoding="UTF-8",
            )
            # Metadados de rastreabilidade
            .withColumn("landinzone_path", F.lit(s3_paths[0]))
            .withColumn("ingestion_datetime", F.to_utc_timestamp(F.current_timestamp(), "UTC"))
        )

        df.printSchema()
        logger.info(
            "  Leitura concluída: %d coluna(s). Escrevendo em '%s'…",
            len(df.columns),
            bronze_table,
        )

        df.writeTo(bronze_table).createOrReplace()

        logger.info("  Tabela '%s' criada/atualizada com sucesso.", bronze_table)
        tables_ok.append(bronze_table)

    except Exception as exc:
        logger.error("  ERRO ao processar tabela '%s': %s", bronze_table, exc, exc_info=True)
        tables_failed.append(bronze_table)

# Resumo
logger.info("=" * 60)
logger.info("Processamento concluído.")
logger.info("  Tabelas OK    : %d — %s", len(tables_ok), tables_ok)
if tables_failed:
    logger.error("  Tabelas FALHA : %d — %s", len(tables_failed), tables_failed)